In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score 
from sklearn import metrics 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
dados = pd.read_csv('C:/Users/abalb/Desktop/seguro_veiculos_trein.csv')

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.describe()

In [ ]:
dados.isnull().any()

In [ ]:
dados.Response.value_counts()

In [ ]:
dados.Response.value_counts().plot.pie(explode=[0,0.1], autopct="%1.1f%%", shadow=True)

In [ ]:
sns.countplot(x=dados['Response'])

In [ ]:
dados2 = dados[dados["Response"]==1]
n_drop= int(.70*len(dados2))
drop_ind = np.random.choice(dados2.index, n_drop, replace=False)
dados_s = dados.drop(drop_ind)
print(dados_s.shape)
dados_s.Response.value_counts().plot.pie(explode=[0,0.1], autopct="%1.1f%%", shadow=True)

In [ ]:
q1 = np.quantile(dados_s.Annual_Premium, 0.25)
q3 = np.quantile(dados_s.Annual_Premium, 0.75)
iqr = q3 - q1

In [ ]:
outlier = dados_s[(dados_s.Annual_Premium < q1-1.5*iqr) | (dados_s.Annual_Premium > q3+1.5*iqr)]
aparado = dados_s[(dados_s.Annual_Premium > q1-1.5*iqr) & (dados_s.Annual_Premium < q3+1.5*iqr)]

In [ ]:
pd.DataFrame({"Dados Final":[len(aparado[aparado.Response==0]), len(aparado[aparado.Response==1])],
              "Outlier":[len(outlier[outlier.Response==0]), len(outlier[outlier.Response==1])]})

In [ ]:
aparado.loc[dados_s.Gender=="Male", "Gender"] = 1
aparado.loc[dados_s.Gender=="Female", "Gender"] = 0
aparado.Gender = aparado.Gender.astype(int)
#
aparado.loc[dados.Vehicle_Age=="< 1 Year", "Vehicle_Age"] = 0
aparado.loc[dados.Vehicle_Age=="1-2 Year", "Vehicle_Age"] = 1
aparado.loc[dados.Vehicle_Age=="> 2 Years", "Vehicle_Age"] = 2
aparado.Vehicle_Age = aparado.Vehicle_Age.astype(int)
#
aparado.loc[dados.Vehicle_Damage=="No", "Vehicle_Damage"] = 0
aparado.loc[dados.Vehicle_Damage=="Yes", "Vehicle_Damage"] = 1
aparado.Vehicle_Damage = aparado.Vehicle_Damage.astype(int)

In [ ]:
aparado.drop(["id"], axis =1, inplace = True)
aparado.Region_Code = aparado["Region_Code"].astype(int)
aparado.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize = (8,8))
sns.heatmap(aparado.corr(), annot=True)
plt.show()

In [ ]:
# modelagem
from sklearn.model_selection import train_test_split
aparado_x, aparado_y = aparado.drop(["Response"], axis=1), aparado["Response"]
#
trein_x, teste_x, trein_y, teste_y = train_test_split(aparado_x, aparado_y, test_size = 0.3)
print(f"Dados Treinamento: {trein_x.shape}\nDados Teste:{teste_x.shape}")

In [ ]:
# Vamos avaliar inicialmente alguns modelos inteligentes com nossos dados
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
def avaliacao (trein_x, teste_x, trein_y, teste_y):
    modelo1 = LogisticRegression().fit(trein_x, trein_y)
    modelo2 = KNeighborsClassifier().fit(trein_x, trein_y)
    modelo3 = GaussianNB().fit(trein_x, trein_y)
    modelo4 = DecisionTreeClassifier().fit(trein_x, trein_y)
    modelo5 = RandomForestClassifier().fit(trein_x, trein_y)
    #
    mod1_aval = f1_score(modelo1.predict(teste_x),teste_y)
    mod2_aval = f1_score(modelo2.predict(teste_x),teste_y)
    mod3_aval = f1_score(modelo3.predict(teste_x),teste_y)
    mod4_aval = f1_score(modelo4.predict(teste_x),teste_y)
    mod5_aval = f1_score(modelo5.predict(teste_x),teste_y)
    #
    print(f"Regr_Logistica: {mod1_aval}\nKNN: {mod2_aval}\nNaive Bayes: {mod3_aval}\nAD: {mod4_aval}\nRF: {mod5_aval}")

In [ ]:
avaliacao(trein_x, teste_x, trein_y, teste_y)

In [ ]:
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state = 0)
us_x, us_y = rus.fit_resample(trein_x, trein_y)
print("Distribuição dos Dados Sobreamostrados Aleatoriamente", Counter(us_y))
avaliacao(us_x, teste_x, us_y, teste_y)

In [ ]:
from imblearn.under_sampling import AllKNN
KNN = AllKNN()
us_x, us_y = KNN.fit_resample(trein_x, trein_y)
print("Distribuição dos Dados Sobreamostrados Aleatoriamente", Counter(us_y))
avaliacao(us_x, teste_x, us_y, teste_y)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
over_sampler = RandomOverSampler(sampling_strategy ='minority')
os_x, os_y = over_sampler.fit_resample(trein_x, trein_y)
print("Distribuição dos Dados Sobreamostrados Aleatoriamente", Counter(us_y))
avaliacao(us_x, teste_x, us_y, teste_y)

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE()
os_x, os_y = smote.fit_resample(trein_x, trein_y)
print("Distribuição dos Dados Sobreamostrados Aleatoriamente", Counter(us_y))
avaliacao(us_x, teste_x, us_y, teste_y)

In [ ]:
us_x, us_y = RandomUnderSampler(random_state = 0, sampling_strategy = 0.3).fit_resample(trein_x, trein_y)
os_x, os_y = SMOTE().fit_resample(us_x, us_y)
avaliacao(os_x, teste_x, os_y, teste_y)

In [ ]:
from imblearn.combine import SMOTEENN
sampled_x, sampled_y = SMOTEENN().fit_resample(trein_x, trein_y)
print("Distribuição SMOTEENN", Counter(sampled_y))
avaliacao(sampled_x, teste_x, sampled_y, teste_y)